### Reads meta data config --> find correct csv file pattern --> use auto loader -- > load only new files --> write to bronze delta table --> update audit log table

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    current_date,
    lit,
    col
)

import uuid
import traceback

dbutils.widgets.text("source_name", "netflix")
dbutils.widgets.text("dataset_name", "netflix_movies")

source_name = dbutils.widgets.get("source_name")
dataset_name = dbutils.widgets.get("dataset_name")
run_id = str(uuid.uuid4())

try:
    config_df = spark.sql(f"""
        SELECT *
        FROM bronze.metadata.bronze_config
        WHERE source_name = '{source_name}'
          AND dataset_name = '{dataset_name}'
          AND is_active = true
    """)

    if config_df.count() == 0:
        raise Exception(f"No active config found for {source_name}.{dataset_name}")

    config = config_df.collect()[0]

    source_path = config["source_path"]
    file_pattern = config["file_pattern"]
    target_table = config["target_table"]
    delimiter = config["delimiter"]
    header_flag = config["header_flag"]

    schema_path = f"/Volumes/bronze/metadata/bronze_schemas/{source_name}/{dataset_name}"
    checkpoint_path = f"/Volumes/bronze/metadata/bronze_checkpoints/{source_name}/{dataset_name}"

    spark.sql(f"""
        INSERT INTO bronze.metadata.bronze_audit_log
        VALUES
        (
          '{run_id}',
          '{source_name}',
          '{dataset_name}',
          '{target_table}',
          'STARTED',
          current_timestamp(),
          NULL,
          NULL
        )
    """)
    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .option("rescuedDataColumn", "_rescued_data")
        .option("header", header_flag)
        .option("delimiter", delimiter)
        .option("inferColumnTypes", "false")
        .option("pathGlobFilter", file_pattern)
        .load(source_path)
        .withColumn("_source_name", lit(source_name))
        .withColumn("_dataset_name", lit(dataset_name))
        .withColumn("_source_file_path", col("_metadata.file_path"))
        .withColumn("_ingestion_timestamp", current_timestamp())
        .withColumn("_ingestion_date", current_date())
        .withColumn("_run_id", lit(run_id))
)

    query = (
        df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(target_table)
    )

    query.awaitTermination()

    spark.sql(f"""
        UPDATE bronze.metadata.bronze_audit_log
        SET status = 'SUCCESS',
            end_time = current_timestamp()
        WHERE run_id = '{run_id}'
    """)

except Exception as e:
    error_message = traceback.format_exc().replace("'", "''")

    spark.sql(f"""
        UPDATE bronze.metadata.bronze_audit_log
        SET status = 'FAILED',
            end_time = current_timestamp(),
            error_message = '{error_message}'
        WHERE run_id = '{run_id}'
    """)

    raise e

In [0]:
%sql
UPDATE bronze.metadata.bronze_config
SET source_path = '/Volumes/bronze/metadata/landing_files/*/'
WHERE source_name = 'netflix';

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:726)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:444)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:444)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:503)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:794)
	at com.data

In [0]:
%sql
UPDATE bronze.metadata.bronze_config
SET file_pattern = 'Netflix TV Shows and Movies.csv'
WHERE dataset_name = 'netflix_tv_shows_movies';

UPDATE bronze.metadata.bronze_config
SET file_pattern = 'Netflix_stock_history.csv'
WHERE dataset_name = 'netflix_stock_history';

UPDATE bronze.metadata.bronze_config
SET file_pattern = 'netflix_movies.csv'
WHERE dataset_name = 'netflix_movies';

UPDATE bronze.metadata.bronze_config
SET file_pattern = 'netflix_reviews.csv'
WHERE dataset_name = 'netflix_reviews';

UPDATE bronze.metadata.bronze_config
SET file_pattern = 'netflix_titles.csv'
WHERE dataset_name = 'netflix_titles';

UPDATE bronze.metadata.bronze_config
SET file_pattern = 'netflix_tv_shows_detailed_up_to_2025 .csv'
WHERE dataset_name = 'netflix_tv_shows_detailed';

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:726)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:444)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:444)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:503)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:794)
	at com.data

In [0]:
%sql
SELECT source_name, dataset_name, source_path, file_pattern, target_table
FROM bronze.metadata.bronze_config
WHERE source_name = 'netflix'
ORDER BY dataset_name; 

source_name,dataset_name,source_path,file_pattern,target_table
netflix,netflix_movies,/Volumes/bronze/metadata/landing_files/*/,netflix_movies.csv,bronze.netflix.netflix_movies
netflix,netflix_reviews,/Volumes/bronze/metadata/landing_files/*/,netflix_reviews.csv,bronze.netflix.netflix_reviews
netflix,netflix_stock_history,/Volumes/bronze/metadata/landing_files/*/,Netflix_stock_history.csv,bronze.netflix.netflix_stock_history
netflix,netflix_titles,/Volumes/bronze/metadata/landing_files/*/,netflix_titles.csv,bronze.netflix.netflix_titles
netflix,netflix_tv_shows_detailed,/Volumes/bronze/metadata/landing_files/*/,netflix_tv_shows_detailed_up_to_2025 .csv,bronze.netflix.netflix_tv_shows_detailed
netflix,netflix_tv_shows_movies,/Volumes/bronze/metadata/landing_files/*/,Netflix TV Shows and Movies.csv,bronze.netflix.netflix_tv_shows_movies


In [0]:
/Volumes/bronze/metadata/landing_files/*/

In [0]:
%sql
select * from bronze.netflix.netflix_movies

title,year,certificate,duration,genre,rating,description,stars,votes,ingest_dt,_rescued_data,_source_name,_dataset_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_run_id,_source_file_path
Cobra Kai,(2018– ),TV-14,30 min,"Action, Comedy, Drama",8.5,"Decades after their 1984 All Valley Karate Tournament bout, a middle-aged Daniel LaRusso and Johnny Lawrence find themselves martial-arts rivals again.","['Ralph Macchio, ', 'William Zabka, ', 'Courtney Henggeler, ', 'Xolo Maridueña']","177,031",2026-02-22,null,netflix,netflix_movies,null,2026-06-07T05:29:30.623Z,2026-06-07,1b4ca34f-98a7-47ea-bce9-e8680d115943,/Volumes/bronze/metadata/landing_files/ingest_dt=2026-02-22/netflix_movies.csv
The Crown,(2016– ),TV-MA,58 min,"Biography, Drama, History",8.7,Follows the political rivalries and romance of Queen Elizabeth II's reign and the events that shaped the second half of the twentieth century.,"['Claire Foy, ', 'Olivia Colman, ', 'Imelda Staunton, ', 'Matt Smith']","199,885",2026-02-22,null,netflix,netflix_movies,null,2026-06-07T05:29:30.623Z,2026-06-07,1b4ca34f-98a7-47ea-bce9-e8680d115943,/Volumes/bronze/metadata/landing_files/ingest_dt=2026-02-22/netflix_movies.csv
Better Call Saul,(2015–2022),TV-MA,46 min,"Crime, Drama",8.9,The trials and tribulations of criminal lawyer Jimmy McGill before his fateful run-in with Walter White and Jesse Pinkman.,"['Bob Odenkirk, ', 'Rhea Seehorn, ', 'Jonathan Banks, ', 'Patrick Fabian']","501,384",2026-02-22,null,netflix,netflix_movies,null,2026-06-07T05:29:30.623Z,2026-06-07,1b4ca34f-98a7-47ea-bce9-e8680d115943,/Volumes/bronze/metadata/landing_files/ingest_dt=2026-02-22/netflix_movies.csv
Devil in Ohio,(2022),TV-MA,356 min,"Drama, Horror, Mystery",5.9,"When a psychiatrist shelters a mysterious cult escapee, her world is turned upside down as the girl's arrival threatens to tear her own family apart.","['Emily Deschanel, ', 'Sam Jaeger, ', 'Gerardo Celasco, ', 'Madeleine Arthur']","9,773",2026-02-22,null,netflix,netflix_movies,null,2026-06-07T05:29:30.623Z,2026-06-07,1b4ca34f-98a7-47ea-bce9-e8680d115943,/Volumes/bronze/metadata/landing_files/ingest_dt=2026-02-22/netflix_movies.csv
Cyberpunk: Edgerunners,(2022– ),TV-MA,24 min,"Animation, Action, Adventure",8.6,"A Street Kid trying to survive in a technology and body modification-obsessed city of the future. Having everything to lose, he chooses to stay alive by becoming an Edgerunner, a Mercenary outlaw also known as a Cyberpunk.","['Zach Aguilar, ', 'Kenichiro Ohashi, ', 'Emi Lo, ', 'Aoi Yûki']","15,413",2026-02-22,null,netflix,netflix_movies,null,2026-06-07T05:29:30.623Z,2026-06-07,1b4ca34f-98a7-47ea-bce9-e8680d115943,/Volumes/bronze/metadata/landing_files/ingest_dt=2026-02-22/netflix_movies.csv
The Sandman,(2022– ),TV-MA,45 min,"Drama, Fantasy, Horror",7.8,"Upon escaping after decades of imprisonment by a mortal wizard, Dream, the personification of dreams, sets about to reclaim his lost equipment.","['Tom Sturridge, ', 'Boyd Holbrook, ', 'Patton Oswalt, ', 'Vivienne Acheampong']","116,358",2026-02-22,null,netflix,netflix_movies,null,2026-06-07T05:29:30.623Z,2026-06-07,1b4ca34f-98a7-47ea-bce9-e8680d115943,/Volumes/bronze/metadata/landing_files/ingest_dt=2026-02-22/netflix_movies.csv
Rick and Morty,(2013– ),TV-MA,23 min,"Animation, Adventure, Comedy",9.2,An animated series that follows the exploits of a super scientist and his not-so-bright grandson.,"['Justin Roiland, ', 'Chris Parnell, ', 'Spencer Grammer, ', 'Sarah Chalke']","502,160",2026-02-22,null,netflix,netflix_movies,null,2026-06-07T05:29:30.623Z,2026-06-07,1b4ca34f-98a7-47ea-bce9-e8680d115943,/Volumes/bronze/metadata/landing_files/ingest_dt=2026-02-22/netflix_movies.csv
Breaking Bad,(2008–2013),TV-MA,49 min,"Crime, Drama, Thriller",9.5,A high school chemistry teacher diagnosed with inoperable lung cancer turns to manufacturing and selling methamphetamine in order to secure his family's future.,"['Bryan Cranston, ', 'Aaron Paul, ', 'Anna Gunn, ', 'Betsy Brandt']","1,831,340",20